# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Group work is allowed

You may work on this lab alone or in a group of up to three. Groups talk through the design together, but **every member turns in their own notebook** — one submission per person, not one per group.

**Every submission must name its group.** Fill in the cell below before you do anything else. If you worked alone, say so there. A notebook that does not name its group is treated as unattributed work, and that is a problem we would both rather avoid.

Your notebooks will look similar to each other, and that is fine when the group is named. It is not fine when it is not.

In [1]:
# Who worked on this notebook?
# List every member, yourself included. Use computing IDs.
# Working alone is fine -- put just your own ID and set SOLO = True.
GROUP_MEMBERS = ["umm4va"]   # e.g. ['abc1de', 'fgh2ij']
SOLO = len(GROUP_MEMBERS) == 1

assert GROUP_MEMBERS != ['your_id_here'], 'Fill in GROUP_MEMBERS before you submit.'
print('Worked alone.' if SOLO else 'Group: ' + ', '.join(GROUP_MEMBERS))

Worked alone.


### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [2]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed
    for x in range(tries):
      try:
        r = requests.get(BASE,params=params,timeout=10)
        if r.status_code == 200:
          return pd.DataFrame(r.json()['daily'])
        if r.status_code not in RETRY_STATUSES:
          raise RuntimeError("not retryable: ",r.status_code)
        last = r.statuscode
      except requests.RequestException as err:
        last = repr(err)
      time.sleep(0.5 * 2 ** x)
    raise RuntimeError(last)

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [3]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False
weather = get_weather("2026-09-05","2026-10-03")
weather_weekend = weather[weather["time"].isin(["2026-09-05","2026-10-03"])].copy()
print(weather_weekend)

          time  temperature_2m_max  precipitation_sum
0   2026-09-05                29.4                2.5
28  2026-10-03                22.0               14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [4]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
if (used_fallback):
  weather_weekend["source"] = "fallback sample"
else:
  weather_weekend["source"] = "open-meteo"

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [5]:
# TODO
wx = weather_weekend[["time","temperature_2m_max","precipitation_sum","source"]].copy()
wx.insert(0,"date",pd.to_datetime(wx["time"]))
wx = wx.drop(columns="time")
wx = wx.rename(columns={"temperature_2m_max": "temp_max","precipitation_sum":"precip_mm"})

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [6]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')
print(wx)

weather table is safe to join.
         date  temp_max  precip_mm      source
0  2026-09-05      29.4        2.5  open-meteo
28 2026-10-03      22.0       14.7  open-meteo


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [7]:
print("2026-10-03 was the wetter day, as it had 14.7 mm of precipitation compared to 2.5 mm")

2026-10-03 was the wetter day, as it had 14.7 mm of precipitation compared to 2.5 mm


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [8]:
from datetime import datetime
params = ['latitude','longitude','start_date','end_date','daily','timezone']
print('Source:  Open-Meteo API')
print('Endpoint:', BASE)
print('Params: ', params)
print('Pulled: ', datetime.now().strftime('%Y-%m-%d %H:%M'))

Source:  Open-Meteo API
Endpoint: https://archive-api.open-meteo.com/v1/archive
Params:  ['latitude', 'longitude', 'start_date', 'end_date', 'daily', 'timezone']
Pulled:  2026-10-17 17:29


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

The values for rain would be NaN for every row, so there would be no data regarding the weather or rain for each day of sales.  However, this may not raise an actual error in the code, so everything may still run with the interpretation that rain has no impact on sales.  Because there is no error, it is possible that nobody catches it.


### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [9]:
#get columns
temp = get_weather("2026-09-05","2026-10-03",lat=28.54,lon=-81.38)
w_orlando = temp[temp["time"].isin(["2026-09-05","2026-10-03"])].copy()
w_orlando["source"] = "open-meteo"

#format columns
wy = w_orlando[["time","temperature_2m_max","precipitation_sum","source"]].copy()
wy.insert(0,"date",pd.to_datetime(wy["time"]))
wy = wy.drop(columns="time")
wy = wy.rename(columns={"temperature_2m_max": "temp_max","precipitation_sum":"precip_mm"})

w_twocities=pd.concat([wx.assign(city="City A"),wy.assign(city="Orlando")],ignore_index=True)
print(w_twocities)

        date  temp_max  precip_mm      source     city
0 2026-09-05      29.4        2.5  open-meteo   City A
1 2026-10-03      22.0       14.7  open-meteo   City A
2 2026-09-05      32.4        3.9  open-meteo  Orlando
3 2026-10-03      30.4       13.4  open-meteo  Orlando


### Before you submit

- `GROUP_MEMBERS` is filled in with real computing IDs, and the cell runs without tripping its assert. This is the one thing people forget.
- Name the same people in the submission comment when you upload.
- Run **Kernel → Restart and Run All** and make sure it still works top to bottom.
- The write-up is answered in your own words, not the group's shared draft.